In [1]:
from transformers import ViTImageProcessor

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
image_mean = processor.image_mean
image_std = processor.image_std
image_size = processor.size['height']

/home/monster/Desktop/codes/ML-Workbook/myvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
print(processor)

ViTImageProcessor {
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}



In [2]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import v2

train_transform = v2.Compose([
    v2.RandomResizedCrop(image_size),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(image_mean, image_std)
])

test_transform = v2.Compose([
    v2.Resize(image_size),
    v2.RandomHorizontalFlip(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(image_mean, image_std)
])

train_data = datasets.FashionMNIST(
    root='../data',
    download=True,
    train=True,
    transform=train_transform
)

test_data = datasets.FashionMNIST(
    root='../data',
    download=True,
    train=False,
    transform=test_transform
)

In [3]:
BATCH_SIZE = 64

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

In [9]:
next(iter(train_loader))

[tensor([[[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           ...,
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]],
 
          [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           ...,
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]],
 
          [[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
           [-

In [4]:
from transformers import ViTForImageClassification
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k')

model.classifier = nn.Linear(in_features=model.classifier.in_features,
                                 out_features = len(train_data.classes))

model.to(device)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1401.68it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out

In [19]:
images, labels = next(iter(train_loader))

In [39]:
# Evaluate the model before finetuning

total_correct = 0
model.eval()
with torch.inference_mode():
    for batch, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        prediction = logits.logits.argmax(-1)
        correct = (prediction == labels).sum().item()
        total_correct += correct
    accuracy = total_correct / len(test_loader.dataset)
    print(f"Test dataset: {len(test_loader.dataset)}")
    print(f'Accuracy: {total_correct} / {len(test_loader.dataset)} = {accuracy * 100:.2f}%')

Test dataset: 10000
Accuracy: 636 / 10000 = 6.36%


In [7]:
def train(model, criterion, optimizer, train_loader, epochs):
    model.train()
    for epoch in range(epochs):
        total_correct, total_loss = 0, 0
        for batch, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            prediction = logits.logits.argmax(-1)
            loss = criterion(logits.logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_correct += (prediction == labels).sum().item()
            total_loss += loss
        train_loss = loss / len(train_loader.dataset)
        train_accuracy = total_correct / len(train_loader.dataset)
        print(f"Epoch: {epoch} \t Train loss: {train_loss:.2f} \t Train Accuracy: {train_accuracy:.2f}")

def evaluate(model):
    total_correct = 0
    model.eval()
    with torch.inference_mode():
        for batch, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            prediction = logits.logits.argmax(-1)
            correct = (prediction == labels).sum().item()
            total_correct += correct
        accuracy = total_correct / len(test_loader.dataset)
        print(f"Test dataset: {len(test_loader.dataset)}")
        print(f'Accuracy: {total_correct} / {len(test_loader.dataset)} = {accuracy * 100:.2f}%')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
num_epochs = 3
train(model, criterion, optimizer, train_loader, num_epochs)
evaluate(model)


Epoch: 0 	 Train loss: 0.00 	 Train Accuracy: 0.23
Epoch: 1 	 Train loss: 0.00 	 Train Accuracy: 0.22
Epoch: 2 	 Train loss: 0.00 	 Train Accuracy: 0.24
Test dataset: 10000
Accuracy: 1686 / 10000 = 16.86%
